# pyBuildingEnergy (Australian revision): define and simulate your own building

Every input the engine uses is set in this notebook. Edit the cells in **Steps 2 to 9**, choose a weather file in **Step 10**, and run **Step 11**.

The values filled in below are the paper's case, Apartment 305 in Carlton, Melbourne. Run everything unchanged first: you should get **131.76 kWh/year**. Then replace the values with your own building.

Units: m, m², W/m²K, °C, W/m². Azimuth: 0 = north, 90 = east, 180 = south, 270 = west. Tilt: 90 = wall, 0 = floor or ceiling.

## Step 1 – Download the engine (once per session)

Run this cell, then click **Runtime → Restart session** once, and continue with the next cell. The restart lets Colab use the package versions the engine was tested with.

In [ ]:
import os
if not os.path.exists("/content/AIB-Pybuildingenergy-Astralia"):
    !git clone https://github.com/samiraghafarigousheh-sys/AIB-Pybuildingenergy-Astralia.git /content/AIB-Pybuildingenergy-Astralia
    !pip install -q -r /content/AIB-Pybuildingenergy-Astralia/engine/requirements.txt

In [ ]:
# Run after every restart
import os, sys
os.chdir("/content/AIB-Pybuildingenergy-Astralia/engine")
sys.path.insert(0, ".")
import pandas as pd
from run import simulate

## Step 2 – General building data

In [ ]:
building = {
    "name": "Apt_305_50_Barry_St_Carlton",
    "country": "Australia",
    "latitude": -37.8,                 # must match the weather file location
    "longitude": 144.968,
    "azimuth_relative_to_true_north": 0,
    "net_floor_area": 20.0,            # conditioned floor area [m²]
    "height": 2.7,                     # floor-to-ceiling height [m]
    "n_floors": 1,
    "wall_thickness": 0.2,             # [m]
    "exposed_perimeter": 0,            # perimeter of a ground floor slab [m]; 0 if no ground contact
    "building_type_class": "Residential_apartment",
    "construction_class": "class_i",   # thermal mass position: class_i, class_e, class_ie, class_d, class_m
    "construction_year": 2006,         # only used to pick a default q50 if you do not set one in Step 7
    "adj_zones_present": True,         # True if you define neighbouring zones in Step 5
    "number_adj_zone": 5,
}

## Step 3 – Site (for wind-driven infiltration)

In [ ]:
site = {
    "terrain_class": "suburban",               # flat_open, open_country, suburban, city_centre
    "floor_level": 3,                          # storey of the zone (raises local wind speed)
    "weather_station_terrain": "open_country", # terrain around the weather station (airports: open_country)
    "weather_station_sensor_height_m": 10.0,
}

## Step 4 – Surfaces (walls, floor, ceiling, windows)

One dictionary per surface. `type` can be:
- `"opaque"`: wall or roof to outdoor air.
- `"transparent"`: window.
- `"adjacent"`: party wall, floor or ceiling to a neighbouring zone. It needs `name_adj_zone`, which must match a zone name in Step 5.
- For a slab on the ground, use `"opaque"` and add `"boundary": "GROUND"`.

Other keys:
- `sky_view_factor`: 0.5 for a vertical outdoor surface, 1.0 for a roof, 0 for internal surfaces.
- `thermal_capacity`: in J/m²K. It is 0 here on purpose for the paper's comparison; use real values for a real building, e.g. heavy masonry 150000–250000.

In [ ]:
surfaces = [
    # ---- Opaque wall to outdoors ----
    {"name": "West exterior wall (opaque)", "type": "opaque",
     "area": 11.88, "u_value": 1.0, "solar_absorptance": 0.75, "thermal_capacity": 0,
     "sky_view_factor": 0.5, "orientation": {"azimuth": 270.0, "tilt": 90.0},
     "name_adj_zone": None, "height": 2.7, "length": 5.0},

    # ---- Party walls, floor and ceiling to neighbouring zones ----
    {"name": "North wall to Apt 306", "type": "adjacent", "name_adj_zone": "apt_north",
     "area": 10.8, "u_value": 2.5, "solar_absorptance": 0.0, "thermal_capacity": 0,
     "sky_view_factor": 0.0, "orientation": {"azimuth": 0.0, "tilt": 90.0}, "height": 2.7, "length": 4.0},
    {"name": "South wall to Apt 304", "type": "adjacent", "name_adj_zone": "apt_south",
     "area": 10.8, "u_value": 2.5, "solar_absorptance": 0.0, "thermal_capacity": 0,
     "sky_view_factor": 0.0, "orientation": {"azimuth": 180.0, "tilt": 90.0}, "height": 2.7, "length": 4.0},
    {"name": "East wall to corridor", "type": "adjacent", "name_adj_zone": "corridor",
     "area": 13.5, "u_value": 2.5, "solar_absorptance": 0.0, "thermal_capacity": 0,
     "sky_view_factor": 0.0, "orientation": {"azimuth": 90.0, "tilt": 90.0}, "height": 2.7, "length": 5.0},
    {"name": "Floor to Apt 205", "type": "adjacent", "name_adj_zone": "apt_below",
     "area": 20.0, "u_value": 1.8, "solar_absorptance": 0.0, "thermal_capacity": 0,
     "sky_view_factor": 0.0, "orientation": {"azimuth": 0.0, "tilt": 0.0}, "height": 5.0, "length": 4.0},
    {"name": "Ceiling to Apt 405", "type": "adjacent", "name_adj_zone": "apt_above",
     "area": 20.0, "u_value": 1.8, "solar_absorptance": 0.0, "thermal_capacity": 0,
     "sky_view_factor": 0.0, "orientation": {"azimuth": 0.0, "tilt": 0.0}, "height": 5.0, "length": 4.0},

    # ---- Windows ----
    {"name": "West window - fixed", "type": "transparent",
     "area": 0.81, "u_value": 5.4, "g_value": 0.65, "glazing_panes": 2,
     "solar_absorptance": 0.5, "thermal_capacity": 0, "sky_view_factor": 0.5,
     "orientation": {"azimuth": 270.0, "tilt": 90.0}, "name_adj_zone": None,
     "height": 0.9, "width": 0.9, "parapet": 1.0, "shading": False},
    {"name": "West window - operable", "type": "transparent",
     "area": 0.81, "u_value": 5.4, "g_value": 0.65, "glazing_panes": 2,
     "solar_absorptance": 0.5, "thermal_capacity": 0, "sky_view_factor": 0.5,
     "orientation": {"azimuth": 270.0, "tilt": 90.0}, "name_adj_zone": None,
     "height": 0.9, "width": 0.9, "parapet": 1.0, "shading": False},
]

## Step 5 – Neighbouring zones

You need one entry for each `name_adj_zone` used in Step 4. The engine holds a `conditioned` neighbour at its `setpoint`.

The `*_elements` lists describe the neighbour's own envelope, one item per element: area [m²], type (`OP` opaque, `W` window), U-value, and orientation (`NV`, `EV`, `SV`, `WV` for vertical walls facing north/east/south/west, `HOR` for horizontal).

If your building has no neighbours, set `adjacent_zones = []` and, in Step 2, set `adj_zones_present` to False and `number_adj_zone` to 0.

In [ ]:
def neighbour(name, azimuth, volume, area, areas, u_values, setpoint=20.0):
    return {"name": name, "orientation_zone": {"azimuth": azimuth},
            "area_facade_elements": areas,
            "typology_elements": ["OP"] * len(areas),
            "transmittance_U_elements": u_values,
            "orientation_elements": ["WV", "NV", "EV", "SV", "HOR", "HOR"],
            "volume": volume, "a_use": area,
            "building_type_class": "Residential_apartment",
            "conditioned": True, "setpoint": setpoint}

apt_box = [13.5, 10.8, 13.5, 10.8, 20.0, 20.0]
adjacent_zones = [
    neighbour("apt_above", 270.0, 54.0, 20.0, apt_box, [1.0, 2.5, 2.5, 2.5, 1.8, 1.8]),
    neighbour("apt_below", 270.0, 54.0, 20.0, apt_box, [1.0, 2.5, 2.5, 2.5, 1.8, 1.8]),
    neighbour("apt_north",   0.0, 54.0, 20.0, apt_box, [2.5, 2.5, 2.5, 2.5, 1.8, 1.8]),
    neighbour("apt_south", 180.0, 54.0, 20.0, apt_box, [2.5, 2.5, 2.5, 2.5, 1.8, 1.8]),
    neighbour("corridor",   90.0, 162.0, 60.0, [81.0, 5.4, 81.0, 5.4, 60.0, 60.0], [2.5] * 6),
]

## Step 6 – Heating/cooling setpoints and plant size

In [ ]:
temperature_setpoints = {
    "heating_setpoint": 18.0, "heating_setback": 15.0,   # [°C] when heating is on / off
    "cooling_setpoint": 26.0, "cooling_setback": 28.0,
}
system_capacities = {"heating_capacity": 1e7, "cooling_capacity": 1e7}   # [W]; very large = ideal (unlimited) system

## Step 7 – Ventilation and air leakage

In [ ]:
ventilation = {
    "ventilation_type": "occupancy",          # occupancy, temp_wind, custom, eplus_infiltration_ext_area, sherman_grimsrud_like
    "flow_rate_per_person": 2.0,              # [l/(s m²)] for 'occupancy'
    "custom_heat_transfer_coefficient_ventilation": None,   # [W/K], only for 'custom'
    "envelope_permeability_q50": 4.0,         # air leakage at 50 Pa [m³/(h m²)]; paper scenarios: 4, 11, 14
    "infiltration_modulation": "stack_wind",  # 'stack_wind' (weather-driven) or 'constant'
    "infiltration_reference_temperature_C": 20.0,
    "infiltration_multiplier_decimals": 6,
}

## Step 8 – Internal gains and their hourly profiles

`full_load` is in W/m² of floor area. The `weekday` and `weekend` lists have 24 fractions (0–1), one per hour starting at midnight.

In [ ]:
internal_gains = [
    {"name": "occupants", "full_load": 4.2,
     "weekday": [1,1,1,1,1,1,1,1,1, .5,.4,.5,.5,.5,.4,.5,.5,.5,.5,.5, 1,1,1,1],
     "weekend": [1,1,1,1,1,1,1,1,1, .8,.7,.7,.7,.7,.5,.5,.7,.8, 1,1,1,1,1,1]},
    {"name": "appliances", "full_load": 3.0,
     "weekday": [.1,.1,.1,.1,.1,.1,.2,.3,.2,.2,.2,.2,.3,.2,.2,.2,.2,.3,.3,.4,1,.6,.4,.2],
     "weekend": [.1,.1,.1,.1,.1,.1,.1,.2,.3,.4,.3,.3,.4,.3,.3,.3,.3,.4,.4,.5,1,.6,.4,.2]},
    {"name": "lighting", "full_load": 0.0,
     "weekday": [0,0,0,0,0,0,.3,.3,.1,0,0,0,0,0,0,0,.1,.5,.8,.8,.8,.7,.4,.1],
     "weekend": [0,0,0,0,0,0,0,.1,.3,.3,.2,.2,.2,.2,.2,.2,.3,.5,.8,.8,.8,.7,.4,.1]},
]

## Step 9 – Operating schedules, climate and model options

In each schedule, 1 = heating/cooling/ventilation available in that hour and 0 = off (the setback temperature applies). There are 24 values, starting at midnight.

In [ ]:
heating_profile = {"weekday": [0]*19 + [1,1,1,1] + [0],          # 19:00–23:00
                   "weekend": [0]*18 + [1,1,1,1,1] + [0]}        # 18:00–23:00
cooling_profile = {"weekday": [0]*17 + [1]*6 + [0],              # 17:00–23:00
                   "weekend": [0]*9 + [1]*14 + [0]}              # 09:00–23:00
ventilation_profile = {"weekday": [1]*24, "weekend": [1]*24}

thermal_bridges = 1.5        # [W/K] total thermal-bridge heat transfer
coldest_month = 7            # 7 = July (southern hemisphere); use 1 for northern

simulation_options = {
    "profile_start_year": 2025,
    "profile_use_holidays": False,
    "dynamic_window_properties": True,       # repaired window physics (paper)
    "dynamic_surface_heat_transfer": True,   # repaired surface coefficients (paper)
    "window_angular_solar_model": "none",    # none, constant, karlsson_roos
    "window_convection_model": "table",      # table, doe2, mowitt, blast, simplecombined
    "external_convection_model": "table",    # table, doe2, mowitt, blast, simplecombined
}

### Assemble the building (no need to edit)

In [ ]:
BUI = {
    "building": building,
    "site": site,
    "adjacent_zones": adjacent_zones,
    "building_surface": surfaces,
    "building_parameters": {
        "temperature_setpoints": temperature_setpoints,
        "system_capacities": system_capacities,
        "ventilation": ventilation,
        "internal_gains": internal_gains,
        "construction": {"wall_thickness": building["wall_thickness"], "thermal_bridges": thermal_bridges},
        "climate_parameters": {"coldest_month": coldest_month},
        "heating_profile": heating_profile,
        "cooling_profile": cooling_profile,
        "ventilation_profile": ventilation_profile,
        "simulation_options": simulation_options,
    },
}
pd.DataFrame(surfaces)[["name", "type", "area", "u_value"]]

## Step 10 – Weather file

Use the included Melbourne file, or upload any `.epw` file, e.g. from climate.onebuilding.org. If you change city, also update `latitude`, `longitude` and `coldest_month` above.

In [ ]:
weather = "weather/AUS_VIC_Melbourne-Essendon.Fields.958660_TMYx.2011-2025.epw"

# To upload your own, uncomment:
# from google.colab import files
# weather = list(files.upload().keys())[0]

## Step 11 – Run

In [ ]:
h = simulate(BUI, weather)

heating, cooling = h.Q_H.sum() / 1000, h.Q_C.sum() / 1000
print(f"Heating: {heating:8.2f} kWh/year  ({heating / building['net_floor_area']:.1f} kWh/m²)")
print(f"Cooling: {cooling:8.2f} kWh/year  ({cooling / building['net_floor_area']:.1f} kWh/m²)")
print(f"Total  : {heating + cooling:8.2f} kWh/year")

In [ ]:
h.index = pd.date_range("2026-01-01", periods=len(h), freq="h")
monthly = h[["Q_H", "Q_C"]].resample("ME").sum() / 1000
monthly.index = monthly.index.strftime("%b")
monthly.plot(kind="bar", ylabel="kWh", title="Monthly heating (Q_H) and cooling (Q_C) need", figsize=(10, 4));

## Step 12 – Save (optional)

Colab deletes files when the session ends, so download what you want to keep.

In [ ]:
import json
json.dump(BUI, open("my_building.json", "w"), indent=2)
h.to_csv("my_building_hourly.csv")
from google.colab import files
files.download("my_building.json"); files.download("my_building_hourly.csv")